# Etapa 3 — Preprocesamiento de datos

**Modelo de Machine Learning para el cribado técnico de reservorios candidatos a waterflooding**

Proyecto de titulación — Maestría en Petróleos, ESPOL
En cooperación con EP Petroecuador (Gerencia de Activo Auca)

---

Este cuaderno corresponde a la **Etapa 3** del flujo de trabajo metodológico:

| Paso | Contenido |
|---|---|
| 3.1 | Limpieza y tratamiento de datos faltantes y valores atípicos |
| 3.2 | División estratificada en entrenamiento y prueba |
| 3.3 | Escalado y selección de características |
| 3.4 | SMOTE solo en entrenamiento, si es necesario |

**Principio que ordena toda la etapa: ninguna transformación aprende del conjunto de prueba.**
Por eso el escalado no se aplica aquí sobre los datos: se define como un transformador *sin ajustar*
que la Etapa 4 colocará dentro de un `Pipeline`, de modo que se ajuste únicamente con los datos de
entrenamiento de cada partición de la validación cruzada. El conjunto de prueba queda retenido hasta
la evaluación final de la Etapa 5.

## 1. Configuración del entorno

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/phabelog/waterflooding-ml-screening.git"
REPO_DIR = "waterflooding-ml-screening"

if os.path.isdir("src/preprocessing"):
    REPO_ROOT = "."
else:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    else:
        # Si ya se clonó en esta sesión, se trae la versión más reciente
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
    REPO_ROOT = REPO_DIR

for sub in ("preprocessing", "analysis", "dataset_generation"):
    sys.path.insert(0, os.path.join(REPO_ROOT, "src", sub))
print("Repositorio listo en:", os.path.abspath(REPO_ROOT))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import preprocess as pp
from generate_dataset import FEATURE_COLUMNS

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
AZUL, ROJO, GRIS = "#1f4e79", "#c0392b", "#7f7f7f"

df = pd.read_csv(os.path.join(REPO_ROOT, "data", "synthetic", "synthetic_dataset.csv"))
print(f"Escenarios: {len(df)} | variables de entrada: {len(FEATURE_COLUMNS)}")

---
## 2. Paso 3.1 — Limpieza: datos faltantes y valores atípicos

In [ ]:
lim = pp.missing_and_duplicates(df)
print(f"Datos faltantes en las variables de entrada: {lim['faltantes_total']}")
print(f"Filas duplicadas:                            {lim['duplicados']}")

El conjunto sintético no contiene datos faltantes ni duplicados: cada escenario se generó completo
y el muestreo continuo hace prácticamente imposible repetir una combinación de 16 variables.

### Valores atípicos

Se aplica el criterio intercuartílico de Tukey (valores más allá de 1.5 veces el rango
intercuartílico). La pregunta relevante no es solo *cuántos* atípicos hay, sino *qué son*: un error
de medición que conviene retirar, o la cola legítima de una distribución asimétrica.

In [ ]:
atip_orig = pp.iqr_outliers(df)
atip_log = pp.iqr_outliers(df, log_columns=pp.LOG_FEATURES)
print("Atípicos en escala original:", atip_orig)
print("Atípicos tras log10:        ", atip_log or "ninguno")
print()
for c in pp.LOG_FEATURES:
    print(f"{c:8s} asimetría original = {df[c].skew():5.2f}  |  tras log10 = {np.log10(df[c]).skew():5.2f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6.5))
etiquetas = {"k_md": "Permeabilidad, k [mD]", "muo_cp": "Viscosidad del petróleo, μo [cP]"}
for fila, c in enumerate(pp.LOG_FEATURES):
    axes[fila, 0].hist(df[c], bins=50, color=GRIS)
    axes[fila, 0].set_title(f"{etiquetas[c]} — escala original", fontsize=9)
    axes[fila, 1].hist(np.log10(df[c]), bins=50, color=AZUL)
    axes[fila, 1].set_title(f"{etiquetas[c]} — escala log10", fontsize=9)
    for ax in axes[fila]:
        ax.set_ylabel("Frecuencia")
plt.tight_layout(); plt.show()

Los 889 valores señalados se concentran exclusivamente en la permeabilidad y la viscosidad del
petróleo, y **desaparecen por completo en escala logarítmica**. No son errores: son la cola derecha de
dos variables que abarcan varios órdenes de magnitud, como ocurre en los reservorios reales. Todos
están dentro de los rangos físicos definidos en la Etapa 1. Por lo tanto **no se elimina ningún
registro**: retirarlos recortaría precisamente los casos de alta permeabilidad y alta viscosidad,
que son parte del dominio que el modelo debe aprender. La asimetría se atiende con la transformación
logarítmica del paso 3.3.

### Tratamiento previsto para los datos de campo

Los datos que entregue EP Petroecuador sí pueden traer valores ausentes (la solicitud pide indicar
"N/D") y valores fuera del rango del entrenamiento. La política adoptada es:

- **Faltantes: no se imputan.** Imputar introduciría supuestos del propio estudio en un conjunto cuyo
  propósito es evaluar el modelo a ciegas. Los registros incompletos se excluyen de la validación y se
  reportan.
- **Fuera de rango: se conservan y se señalan**, porque en ellos el modelo extrapola y su predicción
  debe leerse con cautela.

La función `domain_report` implementa ese control. A modo de demostración se aplica a cinco registros
del conjunto de prueba con dos anomalías introducidas a propósito:

In [ ]:
X_train, X_test, y_train, y_test = pp.split_train_test(df)

ejemplo = X_test.head(5).copy()
ejemplo.iloc[0, ejemplo.columns.get_loc("k_md")] = np.nan    # dato faltante ("N/D")
ejemplo.iloc[1, ejemplo.columns.get_loc("Vdp")] = 0.99       # fuera del rango de entrenamiento
reporte, completos = pp.domain_report(ejemplo, X_train)
print(f"Registros completos: {completos} de {len(ejemplo)}")
reporte[(reporte["faltantes"] > 0) | (reporte["fuera_rango"] > 0)]

---
## 3. Paso 3.2 — División estratificada en entrenamiento y prueba

Se reserva el 20 % de los escenarios como conjunto de prueba. La **estratificación** garantiza la
misma proporción de casos aptos en ambos conjuntos, y la semilla fija (`random_state = 42`) hace que la
división sea idéntica cada vez que se ejecute el código.

El conjunto de prueba queda **retenido**: no participa en el escalado, en la validación cruzada ni en
la optimización de hiperparámetros, y se utiliza una sola vez, en la Etapa 5.

In [ ]:
resumen = pd.DataFrame({
    "escenarios": [len(df), len(X_train), len(X_test)],
    "aptos": [int(df.label.sum()), int(y_train.sum()), int(y_test.sum())],
}, index=["Conjunto completo", "Entrenamiento", "Prueba"])
resumen["% aptos"] = (resumen["aptos"] / resumen["escenarios"] * 100).round(1)
resumen

### Validación cruzada para la Etapa 4

La optimización de hiperparámetros de la Etapa 4 empleará validación cruzada estratificada de 5
particiones **dentro del conjunto de entrenamiento**. Se define aquí para que los cuatro modelos se
comparen exactamente sobre las mismas particiones.

In [ ]:
cv = pp.get_cv()
filas = []
for i, (tr, va) in enumerate(cv.split(X_train, y_train), 1):
    filas.append({"partición": i, "ajuste": len(tr), "validación": len(va),
                  "% aptos en validación": round(y_train.iloc[va].mean() * 100, 1)})
pd.DataFrame(filas).set_index("partición")

---
## 4. Paso 3.3 — Escalado y selección de características

### Escalado

| Variables | Transformación |
|---|---|
| `k_md`, `muo_cp` | log10, luego estandarización |
| resto | estandarización (media 0, desviación 1) |

La transformación logarítmica corrige la asimetría y es físicamente natural: la permeabilidad y la
viscosidad actúan sobre el flujo de forma multiplicativa, a través de cocientes como la relación de
movilidad. La estandarización es indispensable para la SVM, sensible a la escala de las variables;
para los modelos de árboles es irrelevante, pero se aplica a todos para que la comparación de la
Etapa 4 parta de las mismas entradas.

La celda siguiente comprueba que el transformador aprende **solo** del entrenamiento: tras ajustarlo,
el entrenamiento queda exactamente centrado y escalado, mientras que el conjunto de prueba —que no
participó— muestra pequeñas desviaciones, como debe ser.

In [ ]:
pre = pp.build_preprocessor().fit(X_train)
Z_tr = pd.DataFrame(pre.transform(X_train), columns=pre.get_feature_names_out())
Z_te = pd.DataFrame(pre.transform(X_test), columns=pre.get_feature_names_out())
pd.DataFrame({
    "media (entrenamiento)": Z_tr.mean().round(3),
    "desv. (entrenamiento)": Z_tr.std(ddof=0).round(3),
    "media (prueba)": Z_te.mean().round(3),
    "desv. (prueba)": Z_te.std(ddof=0).round(3),
})

### Selección de características

Se conservan las **16 variables de entrada**. La razón de no aplicar una selección estadística
adicional es la siguiente:

- **La redundancia ya se resolvió** en la Etapa 2, con la exclusión de la temperatura.
- **Una selección estadística eliminaría variables físicamente relevantes.** Variables como la
  saturación residual de petróleo o los exponentes de Corey intervienen en las ecuaciones de
  desplazamiento, pero su efecto marginal es débil en el rango muestreado. Débil no es irrelevante, y
  esa distinción corresponde al análisis de importancia de la Etapa 5, no a un filtro previo.
- **Ninguna salida del framework entra al modelo**, conforme a la separación establecida en la Etapa 1.

### Controles negativos

La porosidad y la profundidad **no intervienen en el cálculo del framework**: la porosidad determina
el volumen de petróleo en sitio pero no la fracción recuperable, y la profundidad no entra en ninguna
ecuación ni criterio de cribado. Por construcción, no contienen información sobre la etiqueta.

Se mantienen deliberadamente como **controles negativos**: si el modelo les asignara importancia,
revelaría que aprende ruido del muestreo y no la física del desplazamiento. Además forman parte de los
criterios del cribado tradicional, con el que el modelo se compara en la Etapa 5.

In [ ]:
sel = pp.feature_selection_summary()
print(f"Variables de entrada ({sel['n_variables']}):")
for c in sel["variables"]:
    marca = "  <- control negativo" if c in sel["controles_negativos"] else ""
    marca = marca or ("  <- log10" if c in sel["transformacion_log10"] else "")
    print(f"   {c}{marca}")

---
## 5. Paso 3.4 — Tratamiento del desbalance de clases

La decisión sobre SMOTE se toma con **los mismos umbrales del diagnóstico de la Etapa 2**:

| Clase minoritaria | Tratamiento |
|---|---|
| ≥ 25 % | ponderación de clases, sin generar ejemplos sintéticos |
| < 25 % | SMOTE, aplicado solo sobre el entrenamiento |

In [ ]:
des = pp.resampling_decision(y_train)
print(f"Clase minoritaria en entrenamiento: {des['clase_minoritaria']*100:.1f} %")
print(f"Diagnóstico: {des['diagnostico']}")
print(f"¿Aplicar SMOTE?: {'sí' if des['usar_smote'] else 'no'}")
print()
print("Parámetros de ponderación para la Etapa 4:")
print(f"   class_weight     = '{des['class_weight']}'   (árbol, Random Forest, SVM)")
print(f"   scale_pos_weight = {des['scale_pos_weight']:.3f}        (XGBoost)")

Con una clase minoritaria cercana al 39 %, **SMOTE no se aplica**. Generar ejemplos sintéticos
sobre un conjunto que ya es sintético y está apenas desbalanceado no aportaría información nueva y
podría crear escenarios interpolados sin pasar por el framework físico, es decir, sin garantía de
consistencia. En su lugar, los modelos ponderan las clases, recurso que los cuatro algoritmos admiten
de forma nativa y que no altera los datos.

---
## 6. Conclusiones de la Etapa 3

- **Limpieza.** Sin datos faltantes ni duplicados. Los valores atípicos detectados son la cola de dos
  variables asimétricas, desaparecen en escala logarítmica y se conservan. Para los datos de campo se
  definió una política explícita: los faltantes no se imputan y los valores fuera de rango se señalan.
- **División.** 80 / 20 estratificada y reproducible, con la misma proporción de aptos en ambos
  conjuntos. El conjunto de prueba queda retenido hasta la Etapa 5.
- **Escalado.** Logaritmo en permeabilidad y viscosidad, estandarización en todas las variables,
  entregado sin ajustar para evitar fuga de información.
- **Selección.** Se conservan las 16 variables; porosidad y profundidad actúan como controles
  negativos.
- **Desbalance.** Leve; se atiende con ponderación de clases, sin SMOTE.

---
### Siguiente paso del flujo de trabajo

**Etapa 4 — Desarrollo de modelos:** árbol de decisión como modelo base, Random Forest, XGBoost y SVM,
con optimización de hiperparámetros mediante validación cruzada estratificada sobre las particiones
definidas aquí.

Cuaderno siguiente: `04_desarrollo_modelos.ipynb`